In [0]:
source_path = "/Volumes/instagram/bronzelayer/deloader/"
checkpoint_path = "/Volumes/instagram/bronzelayer/autoloader_checkpoints/"

# 1. Check if the directory has any files before starting
try:
    files = dbutils.fs.ls(source_path)
except Exception:
    files = [] # Handle case where folder doesn't exist yet

if len(files) > 0:
    print(f"🔥 Found {len(files)} files! Starting 'Omnitrix' ingestion...")
    
    # 2. Start the Auto Loader Stream
    raw_df = (spark.readStream
              .format("cloudFiles")
              .option("cloudFiles.format", "csv")
              .option("cloudFiles.inferColumnTypes", "true")
              .option("cloudFiles.schemaLocation", checkpoint_path + "schema")
              .load(source_path))

    # 3. Write to the Table
    (raw_df.writeStream
     .trigger(availableNow=True)
     .option("checkpointLocation", checkpoint_path + "data")
     .option("mergeSchema", "true") 
     .outputMode("append")
     .toTable("instagram.bronzelayer.instagram_users_lifestyle"))
else:
    print("💤 Folder is empty. Skipping ingestion and moving to next task...")